# 03_ROCOv2_Split_Integrity_Audit

Purpose: Independently audit the fixed splits produced by `02_ROCOv2_Dataset_Preparation.ipynb`
before any modeling notebook consumes them.

This notebook does **not** regenerate `train_prepared.csv` / `validation_prepared.csv` /
`test_prepared.csv` / `label_mapping.json`. It only reads them (and the original raw
ROCOv2 metadata + image ZIPs) and verifies:

- The prepared splits still sum to the expected total and match the raw metadata.
- There is no sample/image-ID leakage across Train / Validation / Test.
- No column that could represent a patient/study/case/article-level grouping key
  leaks across splits (screened, not assumed).
- The authoritative 18-label taxonomy in `label_mapping.json` is intact and every
  prepared label column is strictly binary.
- Current split sizes vs. the target 70/15/15 (Train/Calibration/Test) protocol
  specified in the thesis synopsis.
- Every image reference in the prepared metadata actually exists inside the
  corresponding original ZIP archive.

### Dependency
Run `02_ROCOv2_Dataset_Preparation.ipynb` first. This notebook will refuse to run
(`FileNotFoundError`) if its required inputs are missing.

### Naming note
This was originally drafted as a second "Notebook 02" — renumbered to **03** here
since it consumes the outputs of the dataset-preparation notebook and must run after it.


In [1]:
# ============================================================
# CELL 1 — CONFIGURATION AND REPRODUCIBILITY
# ============================================================

import os
import sys
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Google Colab / Google Drive
# ------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive available.")
except ImportError:
    print("Not running in Google Colab.")

# ------------------------------------------------------------
# Explicit paths
# ------------------------------------------------------------
ROCO_ROOT = Path("/content/drive/MyDrive/ROCOv2")
PREPARED_DIR = ROCO_ROOT / "Prepared"
EDA_DIR = ROCO_ROOT / "EDA"
AUDIT_DIR = EDA_DIR / "split_integrity_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Existing prepared metadata (outputs of 02_ROCOv2_Dataset_Preparation.ipynb)
TRAIN_PREPARED_PATH = PREPARED_DIR / "train_prepared.csv"
VALID_PREPARED_PATH = PREPARED_DIR / "validation_prepared.csv"
TEST_PREPARED_PATH = PREPARED_DIR / "test_prepared.csv"

# Existing label mapping
LABEL_MAPPING_PATH = PREPARED_DIR / "label_mapping.json"

# Existing raw concept files
RAW_CONCEPT_FILES = {
    "train": ROCO_ROOT / "Train" / "Concepts" / "train_concepts.csv",
    "validation": ROCO_ROOT / "Validation" / "Concepts" / "valid_concepts.csv",
    "test": ROCO_ROOT / "Test" / "Concepts" / "test_concepts.csv",
}

# Existing image ZIP files
IMAGE_ZIP_FILES = {
    "train": ROCO_ROOT / "Train" / "Images" / "train_images.zip",
    "validation": ROCO_ROOT / "Validation" / "Images" / "valid_images.zip",
    "test": ROCO_ROOT / "Test" / "Images" / "test_images.zip",
}

# ------------------------------------------------------------
# Prepared-metadata column names
# ------------------------------------------------------------
# 02_ROCOv2_Dataset_Preparation.ipynb writes every prepared split with a single
# sample-identifier column named "ID" (see its Cell 10 — create_prepared_split).
# Kept as an explicit mapping (not a hardcoded literal) so later cells stay
# resilient if a split's prepared schema ever diverges.
ID_COLUMNS = {
    "train": "ID",
    "validation": "ID",
    "test": "ID",
}

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Research protocol
# ------------------------------------------------------------
# EXPECTED_TOTAL and CURRENT_COUNTS are DERIVED from the prepared CSVs
# themselves rather than hardcoded. The dataset-preparation notebook can
# legitimately exclude records (empty captions, samples with no CUI matching
# the 18-label taxonomy), which changes these numbers across data-prep runs.
# Hardcoding them here silently drifts out of sync with reality and produces
# false-failure AssertionErrors in Cell 3 — derive them instead so this
# notebook always reflects whatever is actually on disk.
required_for_counts = {
    "train": TRAIN_PREPARED_PATH,
    "validation": VALID_PREPARED_PATH,
    "test": TEST_PREPARED_PATH,
}

_missing_for_counts = [
    f"{name}: {path}"
    for name, path in required_for_counts.items()
    if not path.exists()
]

if _missing_for_counts:
    raise FileNotFoundError(
        "Cannot derive EXPECTED_TOTAL / CURRENT_COUNTS — prepared artifacts "
        "are missing:\n\n" + "\n".join(_missing_for_counts) + "\n\n"
        "Run 02_ROCOv2_Dataset_Preparation.ipynb first."
    )

CURRENT_COUNTS = {
    split_name: len(pd.read_csv(path))
    for split_name, path in required_for_counts.items()
}

EXPECTED_TOTAL = sum(CURRENT_COUNTS.values())

TARGET_RATIO = {
    "train": 0.70,
    "calibration": 0.15,
    "test": 0.15,
}

print("=" * 80)
print("NOTEBOOK 03 — SPLIT INTEGRITY AUDIT")
print("=" * 80)
print("ROCO_ROOT:", ROCO_ROOT)
print("AUDIT_DIR:", AUDIT_DIR)
print("Random seed:", SEED)
print("Current counts (derived from prepared CSVs):", CURRENT_COUNTS)
print("Expected total samples (derived):", EXPECTED_TOTAL)

Mounted at /content/drive
Google Drive available.
NOTEBOOK 03 — SPLIT INTEGRITY AUDIT
ROCO_ROOT: /content/drive/MyDrive/ROCOv2
AUDIT_DIR: /content/drive/MyDrive/ROCOv2/EDA/split_integrity_audit
Random seed: 42
Current counts (derived from prepared CSVs): {'train': 59666, 'validation': 9819, 'test': 9870}
Expected total samples (derived): 79355


## 1. Verify required artifacts exist

Stop rather than silently construct anything from missing inputs.

In [2]:
# ============================================================
# CELL 2 — REQUIRED INPUT ARTIFACT CHECK
# ============================================================

required_inputs = {
    "train_prepared.csv": TRAIN_PREPARED_PATH,
    "validation_prepared.csv": VALID_PREPARED_PATH,
    "test_prepared.csv": TEST_PREPARED_PATH,
    "label_mapping.json": LABEL_MAPPING_PATH,
}

missing_inputs = [
    f"{name}: {path}"
    for name, path in required_inputs.items()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Notebook 03 input artifacts are missing:\n\n"
        + "\n".join(missing_inputs)
        + "\n\n"
        "Run 02_ROCOv2_Dataset_Preparation.ipynb first. "
        "Do not continue until the missing artifacts are generated."
    )

print("PASS — Required prepared metadata artifacts exist.")
for name, path in required_inputs.items():
    print(f"{name:30s}: {path}")

PASS — Required prepared metadata artifacts exist.
train_prepared.csv            : /content/drive/MyDrive/ROCOv2/Prepared/train_prepared.csv
validation_prepared.csv       : /content/drive/MyDrive/ROCOv2/Prepared/validation_prepared.csv
test_prepared.csv             : /content/drive/MyDrive/ROCOv2/Prepared/test_prepared.csv
label_mapping.json            : /content/drive/MyDrive/ROCOv2/Prepared/label_mapping.json


## 2. Load the current split

In [3]:
# ============================================================
# CELL 3 — LOAD CURRENT SPLITS
# ============================================================

train_df = pd.read_csv(TRAIN_PREPARED_PATH)
validation_df = pd.read_csv(VALID_PREPARED_PATH)
test_df = pd.read_csv(TEST_PREPARED_PATH)

split_dfs = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

print("=" * 80)
print("CURRENT SPLIT SIZES")
print("=" * 80)
for split_name, df in split_dfs.items():
    print(
        f"{split_name.capitalize():12s}: "
        f"{len(df):,} rows x {len(df.columns)} columns"
    )

total_current = sum(len(df) for df in split_dfs.values())

print("-" * 80)
print(f"Combined : {total_current:,}")

assert total_current == EXPECTED_TOTAL, (
    f"Expected {EXPECTED_TOTAL:,} total samples, "
    f"but found {total_current:,}."
)

print(f"\nPASS — Combined sample count is {total_current:,} (matches derived EXPECTED_TOTAL).")

CURRENT SPLIT SIZES
Train       : 59,666 rows x 22 columns
Validation  : 9,819 rows x 22 columns
Test        : 9,870 rows x 22 columns
--------------------------------------------------------------------------------
Combined : 79,355

PASS — Combined sample count is 79,355 (matches derived EXPECTED_TOTAL).


## 3. Inspect the actual metadata schema

Do not manually assume `Article_ID`, `Study_ID`, etc. — read the real schema.

In [4]:
# ============================================================
# CELL 4 — ACTUAL METADATA SCHEMA INSPECTION
# ============================================================

print("=" * 80)
print("ACTUAL PREPARED METADATA SCHEMA")
print("=" * 80)

for split_name, df in split_dfs.items():
    print("\n" + "-" * 80)
    print(split_name.upper())
    print("-" * 80)

    schema_df = pd.DataFrame({
        "Column": df.columns,
        "Dtype": [str(df[column].dtype) for column in df.columns],
        "Non_Null": [int(df[column].notna().sum()) for column in df.columns],
        "Unique": [int(df[column].nunique(dropna=True)) for column in df.columns],
    })

    display(schema_df)

ACTUAL PREPARED METADATA SCHEMA

--------------------------------------------------------------------------------
TRAIN
--------------------------------------------------------------------------------


,Column,Dtype,Non_Null,Unique
0,ID,object,59666,59666
1,Image_Member,object,59666,59666
2,Caption,object,59666,59348
3,CUI_String,object,59666,32519
4,label_C0000726,int64,59666,2
5,label_C0002978,int64,59666,2
6,label_C0006141,int64,59666,2
7,label_C0023216,int64,59666,2
8,label_C0024485,int64,59666,2
9,label_C0030797,int64,59666,2



--------------------------------------------------------------------------------
VALIDATION
--------------------------------------------------------------------------------


,Column,Dtype,Non_Null,Unique
0,ID,object,9819,9819
1,Image_Member,object,9819,9819
2,Caption,object,9819,9808
3,CUI_String,object,9819,6548
4,label_C0000726,int64,9819,2
5,label_C0002978,int64,9819,2
6,label_C0006141,int64,9819,2
7,label_C0023216,int64,9819,2
8,label_C0024485,int64,9819,2
9,label_C0030797,int64,9819,2



--------------------------------------------------------------------------------
TEST
--------------------------------------------------------------------------------


,Column,Dtype,Non_Null,Unique
0,ID,object,9870,9870
1,Image_Member,object,9870,9870
2,Caption,object,9870,9857
3,CUI_String,object,9870,6595
4,label_C0000726,int64,9870,2
5,label_C0002978,int64,9870,2
6,label_C0006141,int64,9870,2
7,label_C0023216,int64,9870,2
8,label_C0024485,int64,9870,2
9,label_C0030797,int64,9870,2


## 4. Inspect raw metadata schemas too

The prepared CSV may have discarded an identifier that still exists in the raw files.

In [5]:
# ============================================================
# CELL 5 — RAW ROCOv2 METADATA SCHEMA INSPECTION
# ============================================================

print("=" * 80)
print("RAW ROCOv2 CONCEPT METADATA SCHEMA")
print("=" * 80)

raw_concept_dfs = {}

for split_name, path in RAW_CONCEPT_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Raw concept file missing for {split_name}:\n{path}")

    df = pd.read_csv(path)
    raw_concept_dfs[split_name] = df

    print("\n" + "-" * 80)
    print(split_name.upper())
    print("-" * 80)
    print("Shape:", df.shape)
    print("Columns:")
    for column in df.columns:
        print(
            f"  {column:35s} "
            f"dtype={df[column].dtype} "
            f"unique={df[column].nunique(dropna=True):,}"
        )

RAW ROCOv2 CONCEPT METADATA SCHEMA

--------------------------------------------------------------------------------
TRAIN
--------------------------------------------------------------------------------
Shape: (59958, 2)
Columns:
  ID                                  dtype=object unique=59,958
  CUIs                                dtype=object unique=32,759

--------------------------------------------------------------------------------
VALIDATION
--------------------------------------------------------------------------------
Shape: (9904, 2)
Columns:
  ID                                  dtype=object unique=9,904
  CUIs                                dtype=object unique=6,623

--------------------------------------------------------------------------------
TEST
--------------------------------------------------------------------------------
Shape: (9927, 2)
Columns:
  ID                                  dtype=object unique=9,927
  CUIs                                dtype=object un

## 5. Automatically identify possible grouping fields

This cell does not decide that an identifier is a patient/study/case ID — it only reports candidates.

In [6]:
# ============================================================
# CELL 6 — CANDIDATE GROUPING IDENTIFIER DISCOVERY
# ============================================================

GROUP_KEYWORDS = [
    "patient",
    "patient_id",
    "case",
    "case_id",
    "study",
    "study_id",
    "exam",
    "exam_id",
    "series",
    "series_id",
    "article",
    "article_id",
    "pmc",
    "pmcid",
    "publication",
    "figure",
    "figure_id",
    "accession",
    "accession_id",
]

def candidate_group_columns(columns):
    candidates = []
    for column in columns:
        normalized = (
            str(column)
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
        )
        if any(keyword in normalized for keyword in GROUP_KEYWORDS):
            candidates.append(column)
    return candidates

print("=" * 80)
print("POSSIBLE GROUPING IDENTIFIERS")
print("=" * 80)

for split_name, df in raw_concept_dfs.items():
    candidates = candidate_group_columns(df.columns)
    print(f"\n{split_name.upper()}:")
    if candidates:
        for column in candidates:
            print(f"  Candidate: {column}")
    else:
        print("  No obvious grouping identifier detected from column names.")

print(
    "\nIMPORTANT:"
    "\nCandidate discovery does NOT establish that a field "
    "represents a patient, case, or study."
)

POSSIBLE GROUPING IDENTIFIERS

TRAIN:
  No obvious grouping identifier detected from column names.

VALIDATION:
  No obvious grouping identifier detected from column names.

TEST:
  No obvious grouping identifier detected from column names.

IMPORTANT:
Candidate discovery does NOT establish that a field represents a patient, case, or study.


## 6. Image/sample overlap audit

Directly addresses split leakage at the raw sample-ID level.

In [7]:
# ============================================================
# CELL 7 — IMAGE / SAMPLE ID OVERLAP AUDIT
# ============================================================
# NOTE: this audits the RAW concept metadata (raw_concept_dfs), loaded fresh
# from disk in Cell 5 — not the prepared/exclusion-filtered splits. It is
# validated against a raw-derived total (RAW_EXPECTED_TOTAL), not the
# prepared-derived EXPECTED_TOTAL from Cell 1, since those two numbers can
# legitimately differ once records have been excluded during data prep
# (empty captions, zero-label samples — see excluded_*_records.json).

print("=" * 80)
print("IMAGE / SAMPLE ID OVERLAP AUDIT (raw metadata)")
print("=" * 80)

split_id_sets = {}

for split_name, df in raw_concept_dfs.items():
    if "ID" not in df.columns:
        raise ValueError(
            f"Raw {split_name} concept metadata does not contain an 'ID' column."
        )

    ids = df["ID"].astype(str).str.strip()

    print(
        f"{split_name.capitalize():12s}: "
        f"{len(ids):,} raw IDs ({ids.nunique():,} unique)"
    )

    split_id_sets[split_name] = set(ids)

# ------------------------------------------------------------
# Cross-split intersections
# ------------------------------------------------------------
train_valid_overlap = split_id_sets["train"] & split_id_sets["validation"]
train_test_overlap = split_id_sets["train"] & split_id_sets["test"]
valid_test_overlap = split_id_sets["validation"] & split_id_sets["test"]

print("\n" + "=" * 80)
print("CROSS-SPLIT IMAGE/SAMPLE OVERLAP (raw)")
print("=" * 80)
print("Train <-> Validation:", len(train_valid_overlap))
print("Train <-> Test:", len(train_test_overlap))
print("Validation <-> Test:", len(valid_test_overlap))

assert len(train_valid_overlap) == 0
assert len(train_test_overlap) == 0
assert len(valid_test_overlap) == 0

combined_ids = (
    split_id_sets["train"]
    | split_id_sets["validation"]
    | split_id_sets["test"]
)

RAW_EXPECTED_TOTAL = sum(len(df) for df in raw_concept_dfs.values())
assert len(combined_ids) == RAW_EXPECTED_TOTAL

print("\nPASS — No image/sample ID overlap detected in raw metadata.")
print(f"PASS — Combined raw unique IDs = {len(combined_ids):,}")

if RAW_EXPECTED_TOTAL != EXPECTED_TOTAL:
    print(
        f"\nNOTE — raw total ({RAW_EXPECTED_TOTAL:,}) differs from the "
        f"prepared total ({EXPECTED_TOTAL:,}) by "
        f"{RAW_EXPECTED_TOTAL - EXPECTED_TOTAL:,} records. This is expected "
        f"if the data-preparation notebook excluded records (empty captions, "
        f"zero-label samples) — verified against the exclusion logs in Cell 8."
    )

IMAGE / SAMPLE ID OVERLAP AUDIT (raw metadata)
Train       : 59,958 raw IDs (59,958 unique)
Validation  : 9,904 raw IDs (9,904 unique)
Test        : 9,927 raw IDs (9,927 unique)

CROSS-SPLIT IMAGE/SAMPLE OVERLAP (raw)
Train <-> Validation: 0
Train <-> Test: 0
Validation <-> Test: 0

PASS — No image/sample ID overlap detected in raw metadata.
PASS — Combined raw unique IDs = 79,789

NOTE — raw total (79,789) differs from the prepared total (79,355) by 434 records. This is expected if the data-preparation notebook excluded records (empty captions, zero-label samples) — verified against the exclusion logs in Cell 8.


## 7. Check whether raw and prepared IDs agree

A prepared CSV could have the correct row count but reference the wrong images.

In [8]:
# ============================================================
# CELL 8 — PREPARED <-> RAW ID CONSISTENCY
# ============================================================
# NOTE: prepared metadata is now expected to be a STRICT SUBSET of raw
# metadata, not an exact match, because 02_ROCOv2_Dataset_Preparation.ipynb
# legitimately excludes records (empty captions, samples with no CUI
# matching the 18-label taxonomy) and logs exactly which IDs were dropped.
# This cell checks that every raw ID missing from prepared is accounted for
# by one of those two logs — an UNDOCUMENTED gap is still a real failure.

print("=" * 80)
print("PREPARED <-> RAW SAMPLE ID CONSISTENCY")
print("=" * 80)

SPLIT_NAME_MAP = {"train": "Train", "validation": "Validation", "test": "Test"}

caption_exclusion_path = PREPARED_DIR / "excluded_empty_caption_records.json"
zero_label_exclusion_path = PREPARED_DIR / "excluded_zero_label_records.json"

caption_exclusions = {}
zero_label_exclusions = {}

if caption_exclusion_path.exists():
    with open(caption_exclusion_path, "r", encoding="utf-8") as f:
        caption_exclusions = json.load(f)
else:
    print(f"NOTE — no caption-exclusion log found at {caption_exclusion_path}")

if zero_label_exclusion_path.exists():
    with open(zero_label_exclusion_path, "r", encoding="utf-8") as f:
        zero_label_exclusions = json.load(f)
else:
    print(f"NOTE — no zero-label-exclusion log found at {zero_label_exclusion_path}")

id_consistency_rows = []

for split_name in ["train", "validation", "test"]:
    prepared_df = split_dfs[split_name]
    raw_df = raw_concept_dfs[split_name]
    prepared_id_column = ID_COLUMNS[split_name]
    cap_split = SPLIT_NAME_MAP[split_name]

    if "ID" not in raw_df.columns:
        raise ValueError(f"Raw {split_name} concept metadata does not contain 'ID'.")

    prepared_ids = set(prepared_df[prepared_id_column].astype(str).str.strip())
    raw_ids = set(raw_df["ID"].astype(str).str.strip())

    documented_excluded_ids = (
        set(caption_exclusions.get(cap_split, []))
        | set(zero_label_exclusions.get(cap_split, []))
    )

    missing_from_prepared = raw_ids - prepared_ids
    extra_in_prepared = prepared_ids - raw_ids
    documented_missing = missing_from_prepared & documented_excluded_ids
    undocumented_missing = missing_from_prepared - documented_excluded_ids

    print(f"\n{split_name.upper()}")
    print("Raw IDs:", len(raw_ids))
    print("Prepared IDs:", len(prepared_ids))
    print("Missing from prepared:", len(missing_from_prepared))
    print("  documented (exclusion logs):", len(documented_missing))
    print("  UNDOCUMENTED:", len(undocumented_missing))
    print("Unexpected in prepared:", len(extra_in_prepared))

    assert not extra_in_prepared, (
        f"{split_name}: {len(extra_in_prepared)} unexpected IDs found in "
        f"prepared metadata that do not exist in raw metadata at all."
    )

    assert not undocumented_missing, (
        f"{split_name}: {len(undocumented_missing)} raw IDs are missing from "
        f"prepared metadata but are NOT accounted for by either exclusion "
        f"log. Examples: {sorted(undocumented_missing)[:10]}"
    )

    id_consistency_rows.append({
        "Split": split_name,
        "Raw_IDs": len(raw_ids),
        "Prepared_IDs": len(prepared_ids),
        "Documented_Excluded": len(documented_missing),
        "Undocumented_Missing": len(undocumented_missing),
        "Unexpected_In_Prepared": len(extra_in_prepared),
    })

id_consistency_df = pd.DataFrame(id_consistency_rows)
display(id_consistency_df)

id_consistency_path = AUDIT_DIR / "prepared_raw_id_consistency.csv"
id_consistency_df.to_csv(id_consistency_path, index=False)
print("\nSaved:", id_consistency_path)

print("\nPASS — Every raw ID missing from prepared metadata is accounted for by a documented exclusion.")
print("PASS — No unexpected IDs appear in prepared metadata.")

PREPARED <-> RAW SAMPLE ID CONSISTENCY

TRAIN
Raw IDs: 59958
Prepared IDs: 59666
Missing from prepared: 292
  documented (exclusion logs): 292
  UNDOCUMENTED: 0
Unexpected in prepared: 0

VALIDATION
Raw IDs: 9904
Prepared IDs: 9819
Missing from prepared: 85
  documented (exclusion logs): 85
  UNDOCUMENTED: 0
Unexpected in prepared: 0

TEST
Raw IDs: 9927
Prepared IDs: 9870
Missing from prepared: 57
  documented (exclusion logs): 57
  UNDOCUMENTED: 0
Unexpected in prepared: 0


,Split,Raw_IDs,Prepared_IDs,Documented_Excluded,Undocumented_Missing,Unexpected_In_Prepared
0,train,59958,59666,292,0,0
1,validation,9904,9819,85,0,0
2,test,9927,9870,57,0,0



Saved: /content/drive/MyDrive/ROCOv2/EDA/split_integrity_audit/prepared_raw_id_consistency.csv

PASS — Every raw ID missing from prepared metadata is accounted for by a documented exclusion.
PASS — No unexpected IDs appear in prepared metadata.


## 8. Inspect candidate grouping variables

Now actually examine the candidate identifiers found in Cell 6.

In [10]:
# ============================================================
# CELL 9 — GROUPING VARIABLE AUDIT
# ============================================================

print("=" * 80)
print("GROUPING VARIABLE AUDIT")
print("=" * 80)

group_audit_rows = []

for split_name, df in raw_concept_dfs.items():
    candidates = candidate_group_columns(df.columns)

    for column in candidates:
        series = df[column]
        group_audit_rows.append({
            "Split": split_name,
            "Column": column,
            "Dtype": str(series.dtype),
            "Rows": len(series),
            "Non_Null": int(series.notna().sum()),
            "Unique_Values": int(series.nunique(dropna=True)),
            "Missing_Percentage": float(series.isna().mean() * 100),
        })

group_audit_df = pd.DataFrame(group_audit_rows)

if len(group_audit_df) > 0:
    display(group_audit_df)
    group_audit_path = AUDIT_DIR / "candidate_grouping_variables.csv"
    group_audit_df.to_csv(group_audit_path, index=False)
    print("Saved:", group_audit_path)
else:
    print("No candidate grouping variables were found by schema-based discovery.")

GROUPING VARIABLE AUDIT
No candidate grouping variables were found by schema-based discovery.


**Important**

At this point, do not automatically treat `article_id` (or any similar field) as a
case identifier. For example:

- an article ID may represent a publication;
- one publication can contain multiple figures;
- one figure may not necessarily represent one independent medical case;
- therefore publication-level grouping is not automatically patient/case-level grouping.

Evidence from the actual ROCOv2 metadata semantics is required before any candidate
is used as a case-level grouping key.

## 9. Search all metadata columns for duplicate grouping across splits

Useful even before we know which field is scientifically valid.

In [11]:
# ============================================================
# CELL 10 — CROSS-SPLIT OVERLAP FOR CANDIDATE IDENTIFIERS
# ============================================================

candidate_columns = set()
for df in raw_concept_dfs.values():
    candidate_columns.update(candidate_group_columns(df.columns))

print("=" * 80)
print("CANDIDATE GROUP OVERLAP SCREENING")
print("=" * 80)

overlap_rows = []

for column in sorted(candidate_columns):
    split_sets = {}
    usable = True

    for split_name, df in raw_concept_dfs.items():
        if column not in df.columns:
            usable = False
            break

        values = df[column].dropna().astype(str).str.strip()
        values = values[values.ne("")]
        split_sets[split_name] = set(values)

    if not usable:
        continue

    tv = split_sets["train"] & split_sets["validation"]
    tt = split_sets["train"] & split_sets["test"]
    vt = split_sets["validation"] & split_sets["test"]

    overlap_rows.append({
        "Candidate": column,
        "Train_Unique": len(split_sets["train"]),
        "Validation_Unique": len(split_sets["validation"]),
        "Test_Unique": len(split_sets["test"]),
        "Train_Validation_Overlap": len(tv),
        "Train_Test_Overlap": len(tt),
        "Validation_Test_Overlap": len(vt),
    })

candidate_overlap_df = pd.DataFrame(overlap_rows)
display(candidate_overlap_df)

candidate_overlap_path = AUDIT_DIR / "candidate_group_overlap_screening.csv"
candidate_overlap_df.to_csv(candidate_overlap_path, index=False)
print("Saved:", candidate_overlap_path)

CANDIDATE GROUP OVERLAP SCREENING


""


Saved: /content/drive/MyDrive/ROCOv2/EDA/split_integrity_audit/candidate_group_overlap_screening.csv


Do not interpret a zero overlap here as proof of case-level independence until the meaning of that column has been established.

## 10. Verify the 18-label representation

The verified taxonomy must not be changed by this audit.

In [12]:
# ============================================================
# CELL 11 — 18-LABEL MULTI-HOT INTEGRITY AUDIT
# ============================================================

print("=" * 80)
print("18-LABEL MULTI-HOT INTEGRITY AUDIT")
print("=" * 80)

with open(LABEL_MAPPING_PATH, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

print("Labels in label_mapping.json:", len(label_mapping))

assert len(label_mapping) == 18, (
    "Authoritative label_mapping.json must contain exactly 18 labels."
)

# label_mapping.json (written by 02_ROCOv2_Dataset_Preparation.ipynb) stores
# {CUI: column_index}, not {CUI: canonical_name} — so printing its values
# directly just echoes the index back. This local lookup is DISPLAY-ONLY,
# mirrors the prep notebook's LABEL_TAXONOMY, and does not affect any
# assertion below (those still validate against label_mapping.json / the
# actual prepared columns, not this table).
CANONICAL_LABEL_NAMES = {
    "C0000726": "Abdomen",
    "C0002978": "Angiogram",
    "C0006141": "Breast",
    "C0023216": "Lower Extremity",
    "C0024485": "Magnetic Resonance Imaging",
    "C0030797": "Pelvis",
    "C0032743": "Positron-Emission Tomography",
    "C0037303": "Bone structure of cranium",
    "C0037949": "Vertebral column",
    "C0040405": "X-Ray Computed Tomography",
    "C0041618": "Ultrasonography",
    "C0205106": "Transverse",
    "C0205129": "Sagittal",
    "C0817096": "Chest",
    "C1140618": "Upper Extremity",
    "C1306645": "Plain x-ray",
    "C1996865": "Postero-Anterior",
    "C1999039": "Anterior-Posterior",
}

print("\nLabel mapping:")
for cui, index in sorted(label_mapping.items(), key=lambda kv: kv[1]):
    name = CANONICAL_LABEL_NAMES.get(cui, "(name unavailable)")
    print(f"{index:02d} | {cui} | {name}")

# ------------------------------------------------------------
# Detect label columns
# ------------------------------------------------------------
label_columns = []
for column in train_df.columns:
    if column in label_mapping:
        label_columns.append(column)

# 02_ROCOv2_Dataset_Preparation.ipynb writes label columns as "label_{cui}",
# not bare CUI names — report both possibilities instead of silently assuming.
if len(label_columns) == 18:
    print("\nDetected CUI-named label columns.")
else:
    possible_label_columns = [
        column
        for column in train_df.columns
        if str(column).lower().startswith("label_") or str(column).lower().startswith("label")
    ]
    print("\nCUI-named label columns detected:", len(label_columns))
    print("Possible generic label columns:", possible_label_columns)
    if len(possible_label_columns) == 18:
        label_columns = possible_label_columns
        print("Using generic 'label_*' columns for validation below.")

18-LABEL MULTI-HOT INTEGRITY AUDIT
Labels in label_mapping.json: 18

Label mapping:
00 | C0000726 | Abdomen
01 | C0002978 | Angiogram
02 | C0006141 | Breast
03 | C0023216 | Lower Extremity
04 | C0024485 | Magnetic Resonance Imaging
05 | C0030797 | Pelvis
06 | C0032743 | Positron-Emission Tomography
07 | C0037303 | Bone structure of cranium
08 | C0037949 | Vertebral column
09 | C0040405 | X-Ray Computed Tomography
10 | C0041618 | Ultrasonography
11 | C0205106 | Transverse
12 | C0205129 | Sagittal
13 | C0817096 | Chest
14 | C1140618 | Upper Extremity
15 | C1306645 | Plain x-ray
16 | C1996865 | Postero-Anterior
17 | C1999039 | Anterior-Posterior

CUI-named label columns detected: 0
Possible generic label columns: ['label_C0000726', 'label_C0002978', 'label_C0006141', 'label_C0023216', 'label_C0024485', 'label_C0030797', 'label_C0032743', 'label_C0037303', 'label_C0037949', 'label_C0040405', 'label_C0041618', 'label_C0205106', 'label_C0205129', 'label_C0817096', 'label_C1140618', 'label_C1

## 11. Robust binary-label validation

In [13]:
# ============================================================
# CELL 12 — MULTI-HOT VALUE VALIDATION
# ============================================================

def validate_binary_label_columns(df, columns, split_name):
    if len(columns) != 18:
        raise ValueError(
            f"{split_name}: expected 18 label columns, found {len(columns)}."
        )

    values = df[columns]

    if values.isna().any().any():
        missing_locations = np.argwhere(values.isna().to_numpy())
        raise AssertionError(
            f"{split_name}: NaN values detected in multi-hot labels. "
            f"First locations: {missing_locations[:10].tolist()}"
        )

    unique_values = set(pd.unique(values.to_numpy().ravel()))
    invalid_values = unique_values - {0, 1, 0.0, 1.0, False, True}

    if invalid_values:
        raise AssertionError(
            f"{split_name}: invalid multi-hot values detected: {invalid_values}"
        )

    return True

# Only execute automatically if the 18 columns were detected.
if len(label_columns) == 18:
    for split_name, df in split_dfs.items():
        validate_binary_label_columns(df, label_columns, split_name)
        print(f"PASS — {split_name}: 18-dimensional binary labels.")
else:
    print(
        "STOP HERE for label validation: "
        "the prepared CSV label-column schema needs to be inspected before proceeding."
    )

PASS — train: 18-dimensional binary labels.
PASS — validation: 18-dimensional binary labels.
PASS — test: 18-dimensional binary labels.


## 12. Current split statistics

In [14]:
# ============================================================
# CELL 13 — CURRENT SPLIT STATISTICS
# ============================================================

split_statistics = []

for split_name, df in split_dfs.items():
    count = len(df)
    split_statistics.append({
        "Split": split_name,
        "Samples": count,
        "Percentage": count / EXPECTED_TOTAL * 100,
    })

split_statistics_df = pd.DataFrame(split_statistics)
display(split_statistics_df)

split_statistics_path = AUDIT_DIR / "split_statistics.csv"
split_statistics_df.to_csv(split_statistics_path, index=False)
print("Saved:", split_statistics_path)

,Split,Samples,Percentage
0,train,59666,75.188709
1,validation,9819,12.373511
2,test,9870,12.437780


Saved: /content/drive/MyDrive/ROCOv2/EDA/split_integrity_audit/split_statistics.csv


## 13. Current split versus target 70/15/15

In [15]:
# ============================================================
# CELL 14 — CURRENT VS TARGET SPLIT
# ============================================================

target_counts = {
    "train": EXPECTED_TOTAL * 0.70,
    "calibration": EXPECTED_TOTAL * 0.15,
    "test": EXPECTED_TOTAL * 0.15,
}

comparison_df = pd.DataFrame([
    {
        "Partition": "Train",
        "Current_Count": len(train_df),
        "Current_Percentage": len(train_df) / EXPECTED_TOTAL * 100,
        "Target_Count_70_15_15": round(target_counts["train"]),
        "Target_Percentage": 70.0,
        "Absolute_Count_Difference": len(train_df) - round(target_counts["train"]),
        "Percentage_Point_Difference": len(train_df) / EXPECTED_TOTAL * 100 - 70.0,
    },
    {
        "Partition": "Calibration",
        "Current_Count": len(validation_df),
        "Current_Percentage": len(validation_df) / EXPECTED_TOTAL * 100,
        "Target_Count_70_15_15": round(target_counts["calibration"]),
        "Target_Percentage": 15.0,
        "Absolute_Count_Difference": len(validation_df) - round(target_counts["calibration"]),
        "Percentage_Point_Difference": len(validation_df) / EXPECTED_TOTAL * 100 - 15.0,
    },
    {
        "Partition": "Test",
        "Current_Count": len(test_df),
        "Current_Percentage": len(test_df) / EXPECTED_TOTAL * 100,
        "Target_Count_70_15_15": round(target_counts["test"]),
        "Target_Percentage": 15.0,
        "Absolute_Count_Difference": len(test_df) - round(target_counts["test"]),
        "Percentage_Point_Difference": len(test_df) / EXPECTED_TOTAL * 100 - 15.0,
    },
])

display(comparison_df)

comparison_path = AUDIT_DIR / "current_vs_target_split.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)

if (comparison_df["Absolute_Count_Difference"].abs() > 0).any():
    print(
        "\nNOTE — current split does not match the case-level 70/15/15 "
        "Train/Calibration/Test protocol specified in the thesis synopsis. "
        "This is a reporting-only comparison; this notebook does not repartition data."
    )

,Partition,Current_Count,Current_Percentage,Target_Count_70_15_15,Target_Percentage,Absolute_Count_Difference,Percentage_Point_Difference
0,Train,59666,75.188709,55548,70.0,4118,5.188709
1,Calibration,9819,12.373511,11903,15.0,-2084,-2.626489
2,Test,9870,12.437780,11903,15.0,-2033,-2.562220


Saved: /content/drive/MyDrive/ROCOv2/EDA/split_integrity_audit/current_vs_target_split.csv

NOTE — current split does not match the case-level 70/15/15 Train/Calibration/Test protocol specified in the thesis synopsis. This is a reporting-only comparison; this notebook does not repartition data.


## 14. Check raw image ZIP membership

Verifies that the current split really corresponds to existing images.

In [16]:
# ============================================================
# CELL 15 — RAW IMAGE ZIP MEMBERSHIP AUDIT
# ============================================================

import zipfile

# 02_ROCOv2_Dataset_Preparation.ipynb stores the resolved ZIP member path in a
# column named "Image_Member". Kept as a candidate list (not a hardcoded
# literal) so this still works if that column is ever renamed upstream.
image_column_candidates = [
    "Image_Member",
    "image_member",
    "Image",
    "image",
    "Image_Path",
    "image_path",
    "filename",
    "Filename",
]

zip_audit_rows = []

print("=" * 80)
print("RAW IMAGE ZIP MEMBERSHIP AUDIT")
print("=" * 80)

for split_name, zip_path in IMAGE_ZIP_FILES.items():
    print("\n" + "-" * 80)
    print(split_name.upper())
    print("-" * 80)

    if not zip_path.exists():
        raise FileNotFoundError(f"Image ZIP missing for {split_name}: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        members = set(
            name.replace("\\", "/").lstrip("./")
            for name in zf.namelist()
            if not name.endswith("/")
        )

    print(f"ZIP members: {len(members):,}")

    prepared_df = split_dfs[split_name]

    image_column = next(
        (c for c in image_column_candidates if c in prepared_df.columns),
        None
    )

    if image_column is None:
        print("No explicit image-path column found.")
        zip_audit_rows.append({
            "Split": split_name,
            "Status": "NOT_CHECKED",
            "Reason": "No explicit image-path column detected in prepared metadata.",
        })
        continue

    image_members = prepared_df[image_column].astype(str).str.strip()
    missing_images = [p for p in image_members if p not in members]

    print("Prepared image references:", len(image_members))
    print("Missing from ZIP:", len(missing_images))

    if missing_images:
        display(pd.DataFrame({"Missing_Image": missing_images[:20]}))
        raise FileNotFoundError(
            f"{split_name}: {len(missing_images)} image references "
            f"were not found in the corresponding ZIP."
        )

    zip_audit_rows.append({
        "Split": split_name,
        "Status": "PASS",
        "Prepared_References": len(image_members),
        "Missing": 0,
    })

zip_audit_df = pd.DataFrame(zip_audit_rows)
display(zip_audit_df)

zip_audit_path = AUDIT_DIR / "image_zip_membership_audit.csv"
zip_audit_df.to_csv(zip_audit_path, index=False)
print("\nSaved:", zip_audit_path)

RAW IMAGE ZIP MEMBERSHIP AUDIT

--------------------------------------------------------------------------------
TRAIN
--------------------------------------------------------------------------------
ZIP members: 59,958
Prepared image references: 59666
Missing from ZIP: 0

--------------------------------------------------------------------------------
VALIDATION
--------------------------------------------------------------------------------
ZIP members: 9,904
Prepared image references: 9819
Missing from ZIP: 0

--------------------------------------------------------------------------------
TEST
--------------------------------------------------------------------------------
ZIP members: 9,927
Prepared image references: 9870
Missing from ZIP: 0


,Split,Status,Prepared_References,Missing
0,train,PASS,59666,0
1,validation,PASS,9819,0
2,test,PASS,9870,0



Saved: /content/drive/MyDrive/ROCOv2/EDA/split_integrity_audit/image_zip_membership_audit.csv


## 15. Save image/group overlap report

In [17]:
# ============================================================
# CELL 16 — SPLIT OVERLAP REPORT
# ============================================================

overlap_report = pd.DataFrame([
    {
        "Comparison": "Train <-> Validation",
        "Overlap_Count": len(train_valid_overlap),
        "Status": "PASS" if len(train_valid_overlap) == 0 else "FAIL",
    },
    {
        "Comparison": "Train <-> Test",
        "Overlap_Count": len(train_test_overlap),
        "Status": "PASS" if len(train_test_overlap) == 0 else "FAIL",
    },
    {
        "Comparison": "Validation <-> Test",
        "Overlap_Count": len(valid_test_overlap),
        "Status": "PASS" if len(valid_test_overlap) == 0 else "FAIL",
    },
])

display(overlap_report)

overlap_path = AUDIT_DIR / "split_group_overlap.csv"
overlap_report.to_csv(overlap_path, index=False)
print("Saved:", overlap_path)

,Comparison,Overlap_Count,Status
0,Train <-> Validation,0,PASS
1,Train <-> Test,0,PASS
2,Validation <-> Test,0,PASS


Saved: /content/drive/MyDrive/ROCOv2/EDA/split_integrity_audit/split_group_overlap.csv


## 16. Audit manifest

In [18]:
# ============================================================
# CELL 17 — AUDIT ARTIFACT MANIFEST
# ============================================================

audit_artifact_paths = [
    AUDIT_DIR / "candidate_grouping_variables.csv",
    AUDIT_DIR / "candidate_group_overlap_screening.csv",
    AUDIT_DIR / "prepared_raw_id_consistency.csv",
    AUDIT_DIR / "split_statistics.csv",
    AUDIT_DIR / "current_vs_target_split.csv",
    AUDIT_DIR / "image_zip_membership_audit.csv",
    AUDIT_DIR / "split_group_overlap.csv",
]

manifest = []
for path in audit_artifact_paths:
    manifest.append({
        "Artifact": path.name,
        "Path": str(path),
        "Exists": path.exists(),
        "Size_KB": round(path.stat().st_size / 1024, 2) if path.exists() else None,
    })

audit_manifest_df = pd.DataFrame(manifest)
display(audit_manifest_df)

print("\n" + "=" * 80)
print("NOTEBOOK 03 — SPLIT INTEGRITY AUDIT COMPLETE")
print("=" * 80)
print("\nPROTOCOL:")
print("  - Train / Validation / Test ID overlap: checked, zero found")
print("  - Prepared <-> raw ID consistency: checked")
print("  - Candidate grouping-identifier leakage: screened, not assumed")
print("  - 18-label taxonomy integrity: checked against label_mapping.json")
print("  - Current split vs target 70/15/15 protocol: compared, reported only")
print("  - Image references vs original ZIP contents: checked")
print("  - No prepared CSV, label_mapping.json, or image ZIP was modified.")
print("=" * 80)

,Artifact,Path,Exists,Size_KB
0,candidate_grouping_variables.csv,/content/drive/MyDrive/ROCOv2/EDA/split_integr...,False,NaN
1,candidate_group_overlap_screening.csv,/content/drive/MyDrive/ROCOv2/EDA/split_integr...,True,0.00
2,prepared_raw_id_consistency.csv,/content/drive/MyDrive/ROCOv2/EDA/split_integr...,True,0.16
3,split_statistics.csv,/content/drive/MyDrive/ROCOv2/EDA/split_integr...,True,0.12
4,current_vs_target_split.csv,/content/drive/MyDrive/ROCOv2/EDA/split_integr...,True,0.33
5,image_zip_membership_audit.csv,/content/drive/MyDrive/ROCOv2/EDA/split_integr...,True,0.10
6,split_group_overlap.csv,/content/drive/MyDrive/ROCOv2/EDA/split_integr...,True,0.11



NOTEBOOK 03 — SPLIT INTEGRITY AUDIT COMPLETE

PROTOCOL:
  - Train / Validation / Test ID overlap: checked, zero found
  - Prepared <-> raw ID consistency: checked
  - Candidate grouping-identifier leakage: screened, not assumed
  - 18-label taxonomy integrity: checked against label_mapping.json
  - Current split vs target 70/15/15 protocol: compared, reported only
  - Image references vs original ZIP contents: checked
  - No prepared CSV, label_mapping.json, or image ZIP was modified.


In [19]:
import re

RAW_CAPTION_FILES = {
    "train": ROCO_ROOT / "Train" / "Captions" / "train_captions.csv",
    "validation": ROCO_ROOT / "Validation" / "Captions" / "valid_captions.csv",
    "test": ROCO_ROOT / "Test" / "Captions" / "test_captions.csv",
}

print("=" * 80)
print("CAPTION FILE SCHEMA — checking for anything concepts.csv doesn't have")
print("=" * 80)

for split_name, path in RAW_CAPTION_FILES.items():
    df = pd.read_csv(path)
    print(f"\n{split_name.upper()}: {df.shape}")
    print("Columns:", list(df.columns))

# ------------------------------------------------------------
# Check whether the ID string itself encodes any structure
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("ID STRUCTURE CHECK")
print("=" * 80)

sample_ids = raw_concept_dfs["train"]["ID"].astype(str).head(10).tolist()
print("Sample IDs:", sample_ids)

# ROCOv2 IDs look like: ROCOv2_2023_train_000001 — check whether the numeric
# suffix has any grouping structure (e.g., shared prefix ranges per article)
pattern = re.compile(r"^(.*?)(\d+)$")
prefixes = set()
for _id in raw_concept_dfs["train"]["ID"].astype(str):
    m = pattern.match(_id)
    if m:
        prefixes.add(m.group(1))

print(f"\nDistinct non-numeric ID prefixes in Train: {len(prefixes)}")
print("Sample prefixes:", list(prefixes)[:5])

# ------------------------------------------------------------
# Check whether captions mention PMC/article identifiers as free text
# ------------------------------------------------------------
caption_col = "Caption" if "Caption" in df.columns else "caption"
sample_captions = df[caption_col].head(5).tolist()
print("\nSample captions (checking for embedded article/PMC references):")
for c in sample_captions:
    print(" -", str(c)[:150])

CAPTION FILE SCHEMA — checking for anything concepts.csv doesn't have

TRAIN: (59958, 2)
Columns: ['ID', 'Caption']

VALIDATION: (9904, 2)
Columns: ['ID', 'Caption']

TEST: (9927, 2)
Columns: ['ID', 'Caption']

ID STRUCTURE CHECK
Sample IDs: ['ROCOv2_2023_train_000001', 'ROCOv2_2023_train_000002', 'ROCOv2_2023_train_000003', 'ROCOv2_2023_train_000004', 'ROCOv2_2023_train_000005', 'ROCOv2_2023_train_000006', 'ROCOv2_2023_train_000007', 'ROCOv2_2023_train_000008', 'ROCOv2_2023_train_000009', 'ROCOv2_2023_train_000010']

Distinct non-numeric ID prefixes in Train: 1
Sample prefixes: ['ROCOv2_2023_train_']

Sample captions (checking for embedded article/PMC references):
 - CT chest axial view showing a huge ascending aortic aneurysm (*).
 - Computed tomography (CT) shows floating thrombosis (white arrow)
 - Digitally subtracted angiogram demonstrates active extravasation of the superior rectal artery into the ileal-conduit (blue arrow)
 - Digitally subtracted angiogram of the IMA demonstrat